# One-region parameter simulation


<details>
<summary>References</summary>

* https://github.com/the-virtual-brain/tvb-root/blob/master/tvb_library/tvb/simulator/models/zerlaut.py

* https://github.com/the-virtual-brain/tvb-root/blob/master/tvb_library/tvb/simulator/simulator.py

</details>

## Initialization



Load environment variables

<font color=#fda> NOTE: </font> Prior to loading any modules, we need to load the environment variables from the configuration files. This way, we can configure the TVB modules using environment variables.

De esta forma, podemos configurar los modulos de TVB mediante variables de entorno.

In [ ]:
from dotenv import dotenv_values, find_dotenv
import os
from pathlib import Path

defaults_file = find_dotenv('.env_default', usecwd=True)

if not defaults_file:
    raise FileNotFoundError('Start Jupyter within the repository containing .env_default.')
env_dir = Path(defaults_file).resolve().parent

settings = {**dotenv_values(defaults_file), **dotenv_values(env_dir / '.env')}
settings = {k: v if v is not None else '' for k, v in settings.items() }

os.environ.update(settings)

Modules

In [ ]:
# import json5
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from tvb.basic.neotraits.api import List, NArray
from tvb.datatypes.connectivity import Connectivity
from tvb.simulator.coupling import Linear
from tvb.simulator.integrators import HeunDeterministic
from tvb.simulator.models.zerlaut import ZerlautAdaptationFirstOrder
from tvb.simulator.monitors import Raw
from tvb.simulator.simulator import Simulator

Console

In [ ]:
from rich.console import Console
pprint = Console(force_terminal=True, force_jupyter=False).print

pprint(f'The console is "ready" to print messages with rich formatting.')

Set main directory paths

In [ ]:
def resolve_directory(value, base):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (base / path).resolve()

WORK_DIR = resolve_directory(settings['TVB_WORK_DIR'], env_dir)
RESULT_DIR = resolve_directory(settings['TVB_RESULTS_DIR'], WORK_DIR)

os.makedirs(RESULT_DIR, exist_ok=True)

pprint(f'Working directory: {WORK_DIR}')
pprint(f'Results directory: {RESULT_DIR}')

## Simulation values


In [ ]:
# 1) Entradas de la simulación.

# 1.0) Salida de la simulación.
OUTPUT_FILE = 'single_region_simulation'
OUTPUT_DTYPE = "float32"

In [ ]:
# 1.1) Parámetros de la simulación.
DURATION_MS = 2000.0    # Duración de la simulación en milisegundos.
DT_MS = 0.5             # Paso de integración en milisegundos.

In [ ]:
# 1.2) Parámetros de la entrada. En este caso un pulso.
STIMULUS = {
    "external_input_ex_ex": {
        "pulse": {
            "baseline_hz": 5.0,           # Frecuencia de disparo basal
            "pulse_amplitude_hz": 40.0,   # Amplitud del pulso
            "pulse_onset_ms": 500.0,      # Instante de inicio del pulso
            "pulse_duration_ms": 800.0,   # Duración del pulso
        },
    },
    # # Define los que vas a utilizar
    # "external_input_ex_in": {
    #     "pulse_train": {
    #             "baseline_hz": 1.0,           # Frecuencia de disparo basal
    #             "pulse_amplitude_hz": 10.0,   # Amplitud del pulso
    #             "pulse_onset_ms": 500.0,      # Instante de inicio del pulso
    #             "pulse_duration_ms": 100.0,   # Duración del pulso
    #             "pulse_period_ms": 150.0,     # Periodo del tren de pulsos
    #         },
    # }, 
    
    # "external_input_in_ex": {"external_input_ex_ex": 0.5},
    # "external_input_in_in": {"external_input_ex_in": 0.5},
}


In [ ]:
# 1.3) Parámetros del modelo Zerlaut-AdEx.
MODEL_PARAMETERS = {
    # Población y conectividad
    "N_tot": 10000,
    "p_connect_e": 0.05,
    "p_connect_i": 0.05,
    "g": 0.2,
    "K_ext_e": 400,
    "K_ext_i": 35,
    
    # Parametros de membrana
    "g_L": 10.0,
    "C_m": 200.0,
    "E_L_e": -60.0,
    "E_L_i": -65.0,

    # Parametors de adaptación
    "a_e": 4.0,
    "b_e": 60.0,
    "tau_w_e": 200.0,
    "a_i": 0.0,
    "b_i": 20.0,
    "tau_w_i": 100.0,

    # Parametros sinápticos
    "E_e": 0.0,
    "E_i": -80.0,
    "Q_e": 1.0,
    "Q_i": 5.0,
    "tau_e": 5.0,
    "tau_i": 5.0,
    
    "T": 10.0,
}

# Todos los parámetros son de tipo float menos lo que están en esta lista. 
INTEGER_PARAMETERS = {"N_tot", "K_ext_e", "K_ext_i"}


**Parametros que se utilizaran para estimular el modelo.**

```python
[external_input_ex_ex, external_input_ex_in, 
external_input_in_ex, external_input_in_in]
```

**Parametros de ruido.**

<font color=#fda> NOTE: </font> Vamos a utilizar los valores aleatorios para generar ruido,como aumento de datos. 

```python
[tau_OU, weight_noise]
```

## Build the stimulus

<strong><font color=#fa0>Note:</font></strong> La salida es en KHz


In [ ]:
def _build_pulse(
        time_ms, baseline_hz, pulse_amplitude_hz, pulse_onset_ms, pulse_duration_ms
):
    stimulus = np.full(time_ms.shape, baseline_hz)
    pulse_mask = ((time_ms >= pulse_onset_ms) & 
        (time_ms < pulse_onset_ms + pulse_duration_ms))
    stimulus[pulse_mask] = pulse_amplitude_hz
    
    return stimulus * 1e-3  # Convertir a KHz

In [ ]:
def _build_pulse_train(
        time_ms, baseline_hz, pulse_amplitude_hz, pulse_onset_ms,
        pulse_duration_ms, pulse_period_ms,
):
    stimulus = np.full(time_ms.shape, baseline_hz)
    phase_ms = (time_ms - pulse_onset_ms) % pulse_period_ms
    pulse_mask = ((time_ms >= pulse_onset_ms) & 
        (phase_ms < pulse_duration_ms))
    stimulus[pulse_mask] = pulse_amplitude_hz

    return stimulus * 1e-3  # Convert to kHz

In [ ]:
def build_stimulus(stimulus, duration_ms, dt_ms):
    time_ms = np.arange(0.0, duration_ms, dt_ms)
    output = {}
    for key, value in stimulus.items():
        if value is None: 
            continue

        # ToDo: Por ahora solo se permite una señal de un tipo. 
        #       Crear tipos de estímulos que permitan combinaciones.
        if not isinstance(value, dict) and len(value) != 1:
            raise ValueError(f"Stimulus value for {key} must be a dictionary with one key.")

        match next(iter(value.keys())): # Solo hay una clave en el diccionario.
            case "pulse_train":
                output[key] = _build_pulse_train(time_ms, **value["pulse_train"])
            case "pulse":
                output[key] = _build_pulse(time_ms, **value["pulse"])
        
        # ToDo: Otros tipos de estímulos ...

    for key, value in stimulus.items():
        if value is None:
            continue

        # Ya sabemos que es un diccionario de un solo elemento.
        source, scale = next(iter(value.items())) 

        if source.startswith("external_input_"):
            if source not in output:
                raise ValueError(
                    f"Stimulus value for {key} requires '{source}' to be defined."
                )

            output[key] = scale * output[source]

    return time_ms, output

In [ ]:
time_ms, stimulus = build_stimulus(STIMULUS, DURATION_MS, DT_MS)

In [ ]:
# Plotting

fig, ax = plt.subplots(figsize=(10, 3))

for key, values in stimulus.items():
    ax.plot(time_ms, values, label=key)

ax.set_title("Stimulation Signals")
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Stimulus (Hz)")
ax.legend()
plt.show()

## Simple Simulation

In [ ]:
integrator = HeunDeterministic(dt=DT_MS)
integrator.configure()

In [ ]:
model = ZerlautAdaptationFirstOrder()

for name, value in MODEL_PARAMETERS.items():
    dtype = np.int64 if name in INTEGER_PARAMETERS else np.float64
    setattr(model, name, np.array([value], dtype=dtype))

# Se desactiva el ruido interno para que el ejemplo sea reproducible.
model.weight_noise = np.array([0.0])
model.configure()

In [ ]:
state = np.zeros((5, 1, 1), dtype=np.float64)
coupling = np.zeros((1, 1, 1), dtype=np.float64) # Solo hay un region/nodo
# Nota: El simulador utiza un buffer cirtular, pero aquí no tenemos acceso
history = np.zeros((time_ms.size, 5), dtype=np.float64) 

In [ ]:
def _get_stimulus(step, stimulus):
    return (
        stimulus[k][step] if k in stimulus else None 
            for k in ["external_input_ex_ex", "external_input_ex_in", 
                      "external_input_in_ex", "external_input_in_in"]
    )

def dfunc()
    

In [ ]:


for index in range(time_ms.size - 1):
    # Heun evalúa dfun en t_n y t_{






## 5. Define the parameter-updating simulator

TVB's `Simulator` calls `_loop_update_stimulus` once before every integration step. This one-region subclass applies each source there. The model sees the new values throughout both stages of `HeunDeterministic`. This is the region-only part of `InputParameterSimulator`, included here so the mechanism is visible without importing project code.


In [ ]:
class InputParameterSimulator(Simulator):
    """Update one-region numeric model parameters before each step."""

    input_parameters = List(of=tuple, default=(), label='Time-varying model parameters')

    def configure(self, full_configure=True):
        super().configure(full_configure=full_configure)
        self._parameter_sources = []
        names = set()
        for name, source in self.input_parameters:
            if name in names:
                raise ValueError(f'Duplicate parameter: {name}')
            names.add(name)
            if not isinstance(getattr(type(self.model), name, None), NArray):
                raise ValueError(f'Unknown numeric parameter: {name}')
            if isinstance(source, np.ndarray) and source.ndim == 2 and source.shape[1] == 1:
                kind = 'region_array'
            elif isinstance(source, (list, tuple)) and len(source) == 1 and callable(source[0]):
                kind = 'region_functions'
            else:
                raise ValueError(f'{name} needs a (steps, 1) array or one region function')
            self._parameter_sources.append((name, source, kind))
        return self

    def _loop_update_stimulus(self, step, stimulus):
        super()._loop_update_stimulus(step, stimulus)
        index = step - 1
        time_ms = index * self.integrator.dt
        for name, source, kind in self._parameter_sources:
            value = source[index, 0] if kind == 'region_array' else source[0](time_ms)
            setattr(self.model, name, np.array([[value]], dtype=float))


## 6. Configure the Zerlaut model and one-region connectivity

The model parameters are listed here, next to the model they configure. TVB stores each as a one-element array; population counts are integers. Internal noise is set to zero for a reproducible example. Connectivity has a single zero-weight region, so there is no network input.


In [ ]:
model_parameters = {
    'g_L': 10.0,
    'C_m': 200.0,
    'E_L_e': -60.0,
    'E_L_i': -65.0,
    'a_e': 4.0,
    'b_e': 60.0,
    'tau_w_e': 200.0,
    'a_i': 0.0,
    'b_i': 20.0,
    'tau_w_i': 100.0,
    'E_e': 0.0,
    'E_i': -80.0,
    'Q_e': 1.0,
    'Q_i': 5.0,
    'tau_e': 5.0,
    'tau_i': 5.0,
    'N_tot': 10000,
    'p_connect_e': 0.05,
    'p_connect_i': 0.05,
    'g': 0.2,
    'K_ext_e': 400,
    'K_ext_i': 35,
    'T': 10.0,
}
integer_parameters = {'N_tot', 'K_ext_e', 'K_ext_i'}
model = ZerlautAdaptationFirstOrder()
for name, value in model_parameters.items():
    dtype = np.int64 if name in integer_parameters else np.float64
    setattr(model, name, np.array([value], dtype=dtype))
model.weight_noise = np.array([0.0])
model.configure()
model.variables_of_interest = model.state_variables

connectivity = Connectivity(
    weights=np.zeros((1, 1)),
    tract_lengths=np.zeros((1, 1)),
    centres=np.zeros((1, 3)),
    region_labels=np.array(['region']),
)
print('State variables:', tuple(model.state_variables))


## 7. Configure `InputParameterSimulator`

Set zero coupling, a deterministic Heun integrator, a `Raw` monitor, and a zero initial state for five variables, one region, and one mode.


In [ ]:
simulator = InputParameterSimulator(
    model=model,
    input_parameters=input_parameters,
    connectivity=connectivity,
    coupling=Linear(a=np.array([0.0])),
    integrator=HeunDeterministic(dt=DT_MS),
    monitors=(Raw(),),
    initial_conditions=np.zeros((1, 5, 1, 1)),
).configure()
print('Configured sources:', [(name, kind) for name, _, kind in simulator._parameter_sources])


## 8. Run and restore the initial sample

`Raw` returns `(steps, variables, regions, modes)`. Prepend the known zero initial state so all output series align with the full input grid.


In [ ]:
times, states = simulator.run(n_steps=n_steps)[0]
history = np.vstack((np.zeros((1, 5)), states[:, :, 0, 0]))
result_time_ms = np.r_[0.0, times]
assert states.shape == (n_steps, 5, 1, 1)
assert history.shape == (time_grid_ms.size, 5)
assert np.allclose(result_time_ms, time_grid_ms)
print('Raw states:', states.shape, '| complete history:', history.shape)


## 9. Convert units and save an NPZ

The two firing-rate states are in kHz inside TVB, so multiply them by 1000 for Hz. Adaptation is already in pA. Save the result and enough configuration metadata to identify the run. The default filename differs from the script's output.


In [ ]:
result = {
    'time_ms': result_time_ms,
    'E_hz': history[:, 0] * 1000.0,
    'I_hz': history[:, 1] * 1000.0,
    'W_e_pA': history[:, 2],
    'W_i_pA': history[:, 3],
    'stimulus_hz': stimulus_hz,
    'Fe_ext_hz': fe_ext_hz,
    'Fi_ext_hz': fi_ext_hz,
}
metadata = {
    'simulation': {'duration_ms': DURATION_MS, 'dt_ms': DT_MS, 'method': 'heun'},
    'model': {'name': 'ZerlautAdaptationFirstOrder', 'parameters': model_parameters},
    'external_drive': {'scale': INPUT_SCALE, 'Fi_ratio': FI_RATIO},
    'stimulus': {
        'type': 'pulse', 'baseline_hz': BASELINE_HZ,
        'amplitude_hz': PULSE_AMPLITUDE_HZ,
        'onset_ms': PULSE_ONSET_MS, 'duration_ms': PULSE_DURATION_MS,
    },
    'output': {'path': str(OUTPUT_PATH), 'dtype': OUTPUT_DTYPE},
}
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
np.savez(
    OUTPUT_PATH,
    **{name: np.asarray(values, dtype=OUTPUT_DTYPE) for name, values in result.items()},
    metadata_json=np.array(json.dumps({'config': metadata})),
)
print(f'Saved: {OUTPUT_PATH}')


## 10. Results

The first panel shows the stimulus and four external model parameters in Hz. The other panels show the simulated excitatory and inhibitory firing rates in Hz and adaptation currents in pA. The summary below reports the final and maximum rates.


In [ ]:
with np.load(OUTPUT_PATH, allow_pickle=False) as saved:
    time_ms = saved['time_ms']
    e_hz = saved['E_hz']
    i_hz = saved['I_hz']
    w_e_pa = saved['W_e_pA']
    w_i_pa = saved['W_i_pA']
    plotted_stimulus_hz = saved['stimulus_hz']

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True, constrained_layout=True)
axes[0].plot(time_ms, plotted_stimulus_hz, color='black', label='Stimulus')
for name, source in input_parameters:
    values = source[:, 0] if isinstance(source, np.ndarray) else np.array(
        [source[0](t) for t in time_grid_ms[:-1]]
    )
    axes[0].step(time_grid_ms[:-1], values * 1000.0, where='post', label=name)
axes[0].set_ylabel('External input (Hz)')
axes[0].legend(fontsize=8, ncol=2)

axes[1].plot(time_ms, e_hz, label='E')
axes[1].plot(time_ms, i_hz, label='I')
axes[1].set_ylabel('Firing rate (Hz)')
axes[1].legend()

axes[2].plot(time_ms, w_e_pa, label='W_e')
axes[2].plot(time_ms, w_i_pa, label='W_i')
axes[2].set_ylabel('Adaptation (pA)')
axes[2].set_xlabel('Time (ms)')
axes[2].legend()
for axis in axes:
    axis.grid(alpha=0.25)
plt.show()

print(f'Samples: {time_ms.size} | final E: {e_hz[-1]:.3f} Hz | '
      f'final I: {i_hz[-1]:.3f} Hz')
print(f'Maximum E: {np.max(e_hz):.3f} Hz | maximum I: {np.max(i_hz):.3f} Hz')
